# 4 · Flow fields & networks

Two families: **u/v vector fields** (`vectorfield` arrows, `streamlines`, `barbs`) and **node–edge
networks** (`graph`, `flow`). For the field we use the Lisbon DEM's downhill gradient as a real
`(u, v)` pair; for the network a small synthetic station graph.

**Setup** — derive a coarse `(u, v)` field from the DEM gradient (downsampled so the arrows stay legible).

In [ ]:
import warnings
warnings.filterwarnings("ignore")            # hide backend-mismatch style warnings in the docs

from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import io
from IPython.display import Image
import matplotlib.pyplot as plt
import holoviews as hv
hv.extension("bokeh", "matplotlib")          # Bokeh = interactive; matplotlib = static (projection/barbs)

def show(m, static=False, dpi=96):
    """Display the map — **interactive Bokeh** by default (pan / zoom / hover).

    RUN THIS NOTEBOOK (Run All / Shift-Enter) to see the live interactive maps: a saved .ipynb only
    shows interactive plots once the viewer executes their JavaScript (a running kernel, or a trusted
    notebook). Pass ``static=True`` (used for the matplotlib-only layers — projection, streamlines,
    wind barbs, which have no Bokeh glyph) to render a static PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()                        # interactive Bokeh object

import numpy as np
from pyramids.dataset import Dataset, GeoReference
from digitalearth.interactive import InteractiveMap

dem = Dataset.read_file(str(DATA / "LisbonElevation.tif"))
coarse = dem.resample(dem.cell_size * 12)                # fewer, legible arrows
z = np.nan_to_num(coarse.read_array(band=0).astype("float32"))
dy, dx = np.gradient(z)
u = Dataset.from_array(arr=(-dx).astype("float32"), geo_ref=GeoReference(geo=coarse.geotransform, epsg=coarse.epsg))
v = Dataset.from_array(arr=(dy).astype("float32"), geo_ref=GeoReference(geo=coarse.geotransform, epsg=coarse.epsg))

### `vectorfield` — arrows coloured by magnitude
Interactive arrows; `color_by='magnitude'` (default) colours each by speed. `density` thins the grid (`1.0` keeps every cell).

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="downhill gradient (vectorfield)")
m.vectorfield(u, v, density=0.6, cmap="viridis")
show(m)

### `streamlines` & `barbs` — matplotlib-backend flow
Bokeh has no streamline/wind-barb glyph, so these always render through the matplotlib backend (`barbs` also takes a `density`) — `show(m, static=True)`.

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="streamlines")
m.streamlines(u, v, density=0.5)
show(m, static=True)

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="wind barbs")
m.barbs(u, v, density=0.4)
show(m, static=True)

### `graph` / `flow` — node–edge networks
`graph` draws nodes (a point GeoDataFrame) joined by edges (pairs of node ids, optionally weighted); `flow` is the same with flow-styled edges. Notebooks may build a GeoDataFrame directly (the DX.3 pyramids-only rule applies to the package, not to example notebooks).

In [ ]:
import geopandas as gpd

nodes = gpd.GeoDataFrame(
    {"id": [0, 1, 2, 3]},
    geometry=gpd.points_from_xy([7.0e5, 7.6e5, 8.2e5, 7.6e5], [6.25e6, 6.35e6, 6.30e6, 6.45e6]),
    crs="EPSG:3857",
)
edges = [(0, 1, 5.0), (1, 2, 3.0), (2, 3, 2.0)]

m = InteractiveMap(crs=3857, title="station network (graph)")
m.graph(nodes, edges, weight="weight")
show(m)